## Read the Bronze table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

locations = spark.table(
    "e2e_project.bronze.erp_loc_a101"
)

display(locations)

inspecting

In [0]:
locations.printSchema()
print("Rows:", locations.count())

## First inspect the raw values

In [0]:
display(
    locations.limit(30)
)

In [0]:
display(
    locations
    .groupBy("CNTRY")
    .count()
    .orderBy(F.desc("count"))
)

This is useful because you should actually see what needs normalization before applying rules.

## Trim all string columns

In [0]:
for field in locations.schema.fields:
    if isinstance(field.dataType, StringType):
        locations = locations.withColumn(
            field.name,
            trim(col(field.name))
        )

## Clean the customer identifier

In [0]:
display(
    locations.select("CID").limit(30)
)

standadizing

In [0]:
locations = locations.withColumn(
    "CID",
    F.regexp_replace(
        col("CID"),
        "-",
        ""
    )
)

## Normalize countries

In [0]:
locations = locations.withColumn(
    "CNTRY",
    F.when(
        col("CNTRY") == "DE",
        "Germany"
    )
    .when(
        col("CNTRY").isin("US", "USA"),
        "United States"
    )
    .when(
        (col("CNTRY") == "") |
        col("CNTRY").isNull(),
        "n/a"
    )
    .otherwise(col("CNTRY"))
)

## Validate the normalization

In [0]:
display(
    locations
    .groupBy("CNTRY")
    .count()
    .orderBy("CNTRY")
)

Transformation

      ↓
      
Validation

## Check that hyphens disappeared from IDs

In [0]:
display(
    locations.filter(
        col("CID").contains("-")
    )
)

## Rename the columns

In [0]:
RENAME_MAP = {
    "CID": "customer_number",
    "CNTRY": "country"
}

for old_name, new_name in RENAME_MAP.items():
    locations = locations.withColumnRenamed(
        old_name,
        new_name
    )

## Inspect the final DataFrame

In [0]:
display(locations.limit(20))

In [0]:
locations.printSchema()

## Check duplicate customer numbers

In [0]:
display(
    locations
    .groupBy("customer_number")
    .count()
    .filter(col("count") > 1)
)

## Check NULL customer identifiers

In [0]:
display(
    locations.filter(
        col("customer_number").isNull()
    )
)

Customer location data without an identifier won't be joinable later, so we want to know whether those records exist.

## Write the Silver table

In [0]:
(
    locations.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "e2e_project.silver.erp_customer_location"
    )
)

In [0]:
%sql

SELECT *
FROM e2e_project.silver.erp_customer_location
LIMIT 20;